<a href="https://colab.research.google.com/github/LilaNguyen/Skinterest-Tech-1B/blob/main/Skinterest-Tech-1B/notebooks/skinterest1b_milestone2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Community Segmentation and Categorization Community
Segment customers into distinct skincare communities based on skin concerns, product preferences, and review behavior

## Import Packages

In [2]:
import numpy as np
import pandas as pd
import kagglehub
import os
import re
import nltk

## Load in the Data

In [3]:
# download dataset
root = kagglehub.dataset_download("melissamonfared/sephora-skincare-reviews")

# list files
for dirname, _, filenames in os.walk(root):
  for filename in filenames:
    print(os.path.join(dirname, filename))

# product info
product_df = pd.read_csv(os.path.join(root, 'product_info.csv'))

# load and combine all CSV files
review_files = [file for file in os.listdir(root) if file.startswith('reviews_') and file.endswith('.csv')]
reviews_df = pd.concat([pd.read_csv(os.path.join(root, file)) for file in sorted(review_files)], ignore_index=True)

# merge reviews with product info
merged_df = reviews_df.merge(product_df, on='product_id')

display(merged_df.head())

100%|██████████| 37.0M/37.0M [00:02<00:00, 15.8MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/melissamonfared/sephora-skincare-reviews/versions/1/product_info_skincare.csv
/root/.cache/kagglehub/datasets/melissamonfared/sephora-skincare-reviews/versions/1/product_info.csv
/root/.cache/kagglehub/datasets/melissamonfared/sephora-skincare-reviews/versions/1/reviews_750-1250_masked.csv
/root/.cache/kagglehub/datasets/melissamonfared/sephora-skincare-reviews/versions/1/reviews_1250-end_masked.csv
/root/.cache/kagglehub/datasets/melissamonfared/sephora-skincare-reviews/versions/1/reviews_250-500_masked.csv
/root/.cache/kagglehub/datasets/melissamonfared/sephora-skincare-reviews/versions/1/reviews_500-750_masked.csv
/root/.cache/kagglehub/datasets/melissamonfared/sephora-skincare-reviews/versions/1/reviews_0-250_masked.csv


,Unnamed: 0.1,Unnamed: 0,rating_x,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,0,0,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,...,1,0,0,['Clean at Sephora'],Skincare,Cleansers,NaN,0,NaN,NaN
1,1,1,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
2,2,2,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
3,3,3,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
4,4,4,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data Preprocessing

### Inspect the data

In [5]:
# get dimensions
merged_df.shape

(285412, 45)

In [6]:
# get data types
merged_df.dtypes

,0
Unnamed: 0.1,int64
Unnamed: 0,int64
rating_x,int64
is_recommended,float64
helpfulness,float64
total_feedback_count,int64
total_neg_feedback_count,int64
total_pos_feedback_count,int64
submission_time,object
review_text,object


In [7]:
# display summary stats
merged_df.describe(include='all')

,Unnamed: 0.1,Unnamed: 0,rating_x,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
count,285412.000000,285412.000000,285412.000000,228180.000000,131090.000000,285412.000000,285412.000000,285412.000000,285412,285067,...,285412.000000,285412.000000,285412.000000,263408,285412,285412,231308,285412.000000,129094.000000,129094.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5302,244922,...,NaN,NaN,NaN,381,1,13,23,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-10-21,I love ALL of the Dr. Jart masks so much!!! Th...,...,NaN,NaN,NaN,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Cleansers,Face Wash & Cleansers,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1284,11,...,NaN,NaN,NaN,16138,285412,90702,87926,NaN,NaN,NaN
mean,180616.184242,180616.184242,4.330526,0.843864,0.775019,3.199368,0.685928,2.513440,NaN,NaN,...,0.155470,0.031316,0.300961,NaN,NaN,NaN,NaN,0.827306,41.222140,21.087780
std,168373.481176,168373.481176,1.122987,0.362985,0.320146,20.588738,4.621723,18.311465,NaN,NaN,...,0.362353,0.174171,0.458677,NaN,NaN,NaN,NaN,1.117528,37.336377,13.147197
min,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,3.500000,3.500000
25%,41419.000000,41419.000000,4.000000,1.000000,0.666667,0.000000,0.000000,0.000000,NaN,NaN,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,20.000000,15.000000
50%,115587.500000,115587.500000,5.000000,1.000000,1.000000,0.000000,0.000000,0.000000,NaN,NaN,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,26.000000,18.000000
75%,325059.250000,325059.250000,5.000000,1.000000,1.000000,3.000000,0.000000,2.000000,NaN,NaN,...,0.000000,0.000000,1.000000,NaN,NaN,NaN,NaN,1.000000,51.000000,24.000000


In [8]:
merged_df.head(10)

,Unnamed: 0.1,Unnamed: 0,rating_x,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,0,0,5,1.0,1.00,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,...,1,0,0,['Clean at Sephora'],Skincare,Cleansers,NaN,0,NaN,NaN
1,1,1,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
2,2,2,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
3,3,3,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
4,4,4,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
5,5,5,4,1.0,1.00,1,0,1,2023-03-19,The scent isn’t my favourite but it works grea...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
6,6,6,2,0.0,0.25,8,6,2,2023-03-19,I’ll give this 2 stars for nice packaging and ...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
7,7,7,5,1.0,NaN,0,0,0,2023-03-19,I use this at night or while I’m putting makeu...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
8,8,8,5,1.0,1.00,1,0,1,2023-03-18,I love this stuff. I first had the sample size...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
9,9,9,5,1.0,1.00,2,0,2,2023-03-18,I purchased the Sweet Candy scent at my local ...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0


### Addressing missing values

In [9]:
merged_df = merged_df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0']) # redundant, already have indices col

In [10]:
# dropping features with >50% missing data
percent_missing = merged_df.isnull().mean() * 100
to_drop = percent_missing[percent_missing > 50].index.tolist()
print("Dropping:", to_drop)

merged_df = merged_df.drop(columns=to_drop)

Dropping: ['helpfulness', 'variation_desc', 'value_price_usd', 'sale_price_usd', 'child_max_price', 'child_min_price']


In [11]:
# keep only relevant features
relevant_col = [
    # skin concerns
    'skin_type', 'skin_tone',
    # product preferences
    'primary_category', 'secondary_category', 'brand_name_x',
    'price_usd_x', 'ingredients', 'loves_count', 'limited_edition',
    'sephora_exclusive', 'new',
    # review behavior
    'rating_x', 'is_recommended', 'total_feedback_count',
    'total_pos_feedback_count', 'total_neg_feedback_count',
    'review_text'
]
skin_df = merged_df[relevant_col].copy()

In [12]:
# check missingness in relevant df
skin_df.isnull().sum().sort_values(ascending=False)

,0
is_recommended,57232
skin_tone,55226
skin_type,36874
ingredients,3731
review_text,345
brand_name_x,0
price_usd_x,0
secondary_category,0
primary_category,0
limited_edition,0


In [13]:
# fill missing values
skin_df['is_recommended'] = skin_df['is_recommended'].fillna(skin_df['is_recommended'].mode()[0]) # fill with most common value
skin_df['skin_tone'] = skin_df['skin_tone'].fillna('Unknown') # fill with unknown to prevent introducing bias
skin_df['skin_type'] = skin_df['skin_type'].fillna('Unknown') # same reasoning as above
skin_df['ingredients'] = skin_df['ingredients'].fillna('')
skin_df['review_text'] = skin_df['review_text'].fillna('')

### Addressing outliers

### Normalize numerical features

### Standardize and scale parameters

### Feature engineer categorical variables

In [14]:
# one-hot encode
encoded_df = pd.get_dummies(skin_df,
                            columns=['skin_type', 'skin_tone', 'primary_category', 'secondary_category', 'brand_name_x']
                            )
encoded_df

,price_usd_x,ingredients,loves_count,limited_edition,sephora_exclusive,new,rating_x,is_recommended,total_feedback_count,total_pos_feedback_count,...,brand_name_x_Wander Beauty,brand_name_x_Wishful,brand_name_x_Youth To The People,brand_name_x_alpyn beauty,brand_name_x_belif,brand_name_x_fresh,brand_name_x_goop,brand_name_x_iNNBEAUTY PROJECT,brand_name_x_innisfree,brand_name_x_tarte
0,19.0,"['Water (Aqua), Dipropylene Glycol, Peg-6 Capr...",177,0,0,0,5,1.0,2,2,...,False,False,False,False,False,False,False,False,False,False
1,24.0,"['Diisostearyl Malate, Hydrogenated Polyisobut...",1081315,0,1,0,1,0.0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,24.0,"['Diisostearyl Malate, Hydrogenated Polyisobut...",1081315,0,1,0,5,1.0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,24.0,"['Diisostearyl Malate, Hydrogenated Polyisobut...",1081315,0,1,0,5,1.0,0,0,...,False,False,False,False,False,False,False,False,False,False
4,24.0,"['Diisostearyl Malate, Hydrogenated Polyisobut...",1081315,0,1,0,5,1.0,0,0,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285407,32.0,"['Aloe Barbadensis Leaf Juice, Hexylene Glycol...",14632,0,1,0,5,1.0,0,0,...,False,False,False,False,False,False,False,False,False,False
285408,32.0,"['Aloe Barbadensis Leaf Juice, Hexylene Glycol...",14632,0,1,0,5,1.0,0,0,...,False,False,False,False,False,False,False,False,False,False
285409,32.0,"['Aloe Barbadensis Leaf Juice, Hexylene Glycol...",14632,0,1,0,5,1.0,0,0,...,False,False,False,False,False,False,False,False,False,False
285410,32.0,"['Aloe Barbadensis Leaf Juice, Hexylene Glycol...",14632,0,1,0,5,1.0,0,0,...,False,False,False,False,False,False,False,False,False,False


## Making the Model

In [15]:
# consider using K-means or hierarchical clustering

## Testing the Model

## Model Validation

## Visualizations